# 13.7 偏好資料會不會把長度當成捷徑？


同題「2+2=?」，短版「4」和長版「4。」重複八次，只有同一個事實。如果大多數較佳回答恰好都是長版，模型可能學到增加文字量，而非增加有用解釋。本節先數兩邊有哪些位置，將這個隱藏差異記下來。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import ByteTokenizer

tok = ByteTokenizer()
answers = ["4", "4。" * 8]
for answer in answers:
    body = tok.encode(answer)
    print("回答", answer, "正文位元組tokens", len(body), "含結束學習位置", len(body) + 1)

預設ByteTokenizer按UTF-8位元組分片：數字4佔1，中文句號佔3。短正文1個token、長正文32個；加EOS後，有效目標是2與33。這只數資料，沒有測模型偏好。

長度與品質一起變，讓原因難分，這叫**混淆因素**。可以增加長短接近但品質不同的比較，也讓「短而完整」勝過「長而重複」。需要解釋的任務則保留有用長答，不把短一律標好。

13.3採整段log加總；改平均會改數學目標，不能當成無影響的修正。每篇自己的長度同時進入策略與參考計算，參考差分也不保證消除資料的長度偏差。

練習把重複次數改2，預測正文8、含EOS9。接著用「兩組各2塊積木合起來，共有4塊」作長版：在只回數字的要求下它不合格，在要求分組說明時則有需要的資訊。偏好理由應跟著請求，不能只引用長度。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

正式實驗另做一支只比較附加句的控制：同一題`2+2=?`，指定較佳為`4`、較差為`4; answer complete`，兩邊數值都正確。第一篇含EOS有兩個目標，第二篇有19個；本支線的偏好理由是不要附加句，並不是第二篇算錯。起點在七道最後檢查上，本來就給短候選較高機率。更新200次後，七道相對差距都改善，短候選仍都排前，不是從零題變成七題；但是七道自由生成仍零題答對，上面這題實際生成`3`。這讓我們把格式偏好與算術能力分開觀察，不能把7／7排序當成加法通過。

相對差距增加，甚至不要求較佳候選自己的機率上升。在驗證題`0+3=?`上，正確短答`3`的log機率從約−13.30943降為−13.37556，卻仍有約115.77823的相對差距改善，因為附加句版本下降更多。這個數字是兩篇完整機率與參考差距的變化，不是模型算對的信心。長短、答案內容與生成結果需要一起回查；這支線沒有消除所有長度偏差，也沒有改成平均分數。

</details>
